# ch10 RAG 检索增强（练习）

> 把 `____` 换成正确的代码。每道题下方的 `assert` 会立刻告诉你对不对。

**本章目标**：学会把"检索效果不好"这个模糊的问题，
拆成**可度量、可归因、可定向改进**的一串实验。

## 四条实测结论

| # | 动作 | MRR 变化 |
|---|---|---|
| ① | 加深清洗（去 URL、去字间空格） | 0.5659 → **0.6111** |
| ② | 误差归因后的定向改进（设备过滤 + 缺陷词加权） | 0.6111 → **0.9345** |
| ③ | 换算法 / 融合 / 重排 | ±0.02 |
| ④ | 剔掉索引里的无效占位符 | 约 ±0.01 |

**改进顺序**：修数据 → 定口径 → 看归因 → 定向改进 → 最后才调算法。

## 本章地图

| 任务 | 内容 |
|---|---|
| 1~2 | 清洗深度实验 / 无效占位符与 `avgdl` |
| 3~4 | 手写 BM25（idf + 单条打分） |
| 5 | 分层切分与正样本规模 |
| 6~10 | 四条路线 / 指标拆解 / SVD 维度 / 参数扫描 / 融合重排 |
| 11~12 | 误差归因四象限 → 定向改进 |
| 13~14 | 拒答阈值 / prompt 拼装与引用 |

In [ ]:
import math
import re
from collections import Counter

import jieba
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

jieba.setLogLevel(60)          # 关掉 jieba 的日志，输出才干净
jieba.initialize()
np.random.seed(42)
SEED = 42

## 脚手架（已给好，直接调用）

清洗函数、分词器、BM25 类、四个指标、正样本定义都在这里 —— 全章统一口径。

In [ ]:
RAW = pd.read_csv("data/synth_text.csv", keep_default_na=False)

# ---- 清洗用的正则（与 ch01 同一套口径）----
TAG_RE = re.compile(r"<[^>]+>")                       # HTML 标签
STATION_RE = re.compile(r"STATION_[A-Z]_\d{2}")       # 台区编号（st1~st3 的语义噪声）
URL_RE = re.compile(r"https?://\S+|10\.0\.0\.1/\S*")   # 附件 URL
PUNCT = set("，,。.、；;：:（）()<>《》“”‘’！!？?、·-—[]{}【】 　/")
STOP = {"请", "的", "了", "和", "与", "及", "或", "位于", "存在",
        "发现", "详见", "一般", "严重", "危急"}
BAD_PLACEHOLDERS = {"N/A", "-", "无"}                  # 无效占位符（不是真文本）

# ---- 领域词表：误差归因之后要加权的那两族词 ----
DEV_WORDS = {"变压器", "断路器", "隔离开关", "避雷器", "电流互感器", "绝缘子", "套管",
             "母线", "互感器", "开关", "隔离"}
TYP_WORDS = {"渗油", "漏油", "油迹", "油污", "滴油", "锈蚀", "生锈", "铁锈", "破损",
             "裂纹", "碎裂", "缺口", "异物", "鸟巢", "塑料袋", "树枝", "发热",
             "过热", "烫手", "温度", "偏高", "温升"}


def clean_basic(text):
    """A 口径：只去 HTML 标签与台区编号，空白折叠成一个空格。"""
    return re.sub(r"\s+", " ", STATION_RE.sub(" ", TAG_RE.sub(" ", text))).strip()


def clean_full(text):
    """B 口径：再去 URL，并且**去掉全部空白** —— 顺手修掉"字间空格"这类 OCR 噪声。"""
    return re.sub(r"\s+", "", STATION_RE.sub(" ", URL_RE.sub(" ", TAG_RE.sub(" ", text))))


def tok(text):
    """分词 + 去标点 + 去停用词。"""
    return [w for w in jieba.lcut(text) if w.strip() and w not in PUNCT and w not in STOP]


def tokenize_all(texts):
    return [tok(t) for t in texts]


def load_corpus():
    """从 CSV 到"可检索语料"：清洗 → 丢空文本 → 拼出分组键。"""
    df = RAW.copy()
    df["text"] = df["缺陷描述"].map(clean_full)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    df["group"] = df["设备类型"] + "|" + df["缺陷类型"]
    return df


def stratified_split(df, frac=0.1, seed=42):
    """按 `设备类型|缺陷类型` 分层抽查询 —— 每组抽 round(n × frac) 条，至少 1 条。

    为什么要分层：如果随机抽，小分组可能一条都不进查询集（或全被抽走），
    指标就变成"大分组说了算"。分层保证每个分组都有自己的查询。
    """
    rng = np.random.default_rng(seed)
    picked = []
    for _, sub in df.groupby("group"):
        picked.extend(rng.choice(sub.index.to_numpy(),
                                 size=max(1, round(len(sub) * frac)), replace=False).tolist())
    picked = sorted(int(i) for i in picked)
    return df.drop(index=picked).reset_index(drop=True), df.loc[picked].reset_index(drop=True)


def bm25_score(q_tokens, tf_doc, idf, dl, avgdl, k1=1.5, b=0.75):
    """单文档 BM25 分数 —— 本章公式的**唯一真相**。

    三个部分相乘再求和：
        idf(w)                稀有词更值钱（概率型 idf）
        f·(k1+1)/(f + k1·…)   tf 饱和：一个词出现 1 次和 100 次不该差 100 倍
        分母里的 dl/avgdl     长文档天然更容易撞词，要按长度罚一下
    """
    s = 0.0
    for w in q_tokens:
        f = tf_doc.get(w, 0)
        if f:
            s += idf[w] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))
    return s


class BM25:
    """手写 BM25：全库 idf + 逐文档 tf + 长度归一化。"""

    def __init__(self, doc_tokens, k1=1.5, b=0.75):
        self.docs = doc_tokens
        self.k1, self.b = k1, b
        self.N = len(doc_tokens)
        self.avgdl = sum(len(d) for d in doc_tokens) / self.N
        self.tf = [Counter(d) for d in doc_tokens]
        dfreq = Counter()
        for d in doc_tokens:
            dfreq.update(set(d))          # 注意：这里是 set(d)，统计的是"含该词的文档数"
        self.dfreq = dfreq
        self.idf = {w: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for w, n in dfreq.items()}

    def score(self, q_tokens, i):
        return bm25_score(q_tokens, self.tf[i], self.idf,
                          len(self.docs[i]), self.avgdl, self.k1, self.b)

    def rank(self, q_tokens):
        return order_desc([self.score(q_tokens, i) for i in range(self.N)])


def weighted_score(q_tokens, doc_idx, boost_words, boost=2.0):
    """在 BM25 基础上给指定词族加权（误差归因之后才用得上的武器）。"""
    tf_doc = BM.tf[doc_idx]
    dl = len(BM.docs[doc_idx])
    s = 0.0
    for w in q_tokens:
        f = tf_doc.get(w, 0)
        if f:
            w_boost = boost if w in boost_words else 1.0
            s += w_boost * BM.idf[w] * f * (BM.k1 + 1) / (
                f + BM.k1 * (1 - BM.b + BM.b * dl / BM.avgdl))
    return s


def order_desc(scores):
    """按分数降序给出文档下标。"""
    return np.argsort(np.asarray(scores))[::-1].tolist()


def l2_normalize(matrix):
    """行向量 L2 归一化 —— 稠密向量算余弦相似度前必做（0 范数行保持 0）。"""
    norm = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norm, out=np.zeros_like(matrix), where=norm > 0)


def rrf_fuse(rank_lists, k=60, top=100):
    """Reciprocal Rank Fusion：只看名次、不看分数，所以天然免疫"两路分数量纲不同"。"""
    agg = {}
    for ranks in rank_lists:
        for pos, doc in enumerate(ranks[:top], 1):
            agg[doc] = agg.get(doc, 0.0) + 1.0 / (k + pos)
    return sorted(agg, key=lambda doc: -agg[doc])


def relevant_sets(docs, queries):
    """正样本 = 与查询同属「设备类型|缺陷类型」组的全部文档（**多正样本**）。"""
    return [set(docs.index[docs["group"] == g]) for g in queries["group"]]


def recall_at(ranks, rel_sets, k):
    return float(np.mean([len(set(r[:k]) & rel) / len(rel) for r, rel in zip(ranks, rel_sets)]))


def hit_at(ranks, rel_sets, k):
    return float(np.mean([1.0 if set(r[:k]) & rel else 0.0 for r, rel in zip(ranks, rel_sets)]))


def reciprocal_rank(ranks, rel_sets):
    return float(np.mean([next((1 / pos for pos, d in enumerate(r, 1) if d in rel), 0.0)
                          for r, rel in zip(ranks, rel_sets)]))


def ndcg_at(ranks, rel_sets, k=10):
    """nDCG@k：越靠前命中越值钱；分母是"理想排序"的 DCG。"""
    gains = []
    for r, rel in zip(ranks, rel_sets):
        dcg = sum(1 / math.log2(pos + 1) for pos, d in enumerate(r[:k], 1) if d in rel)
        idcg = sum(1 / math.log2(pos + 1) for pos in range(1, min(k, len(rel)) + 1))
        gains.append(dcg / idcg if idcg else 0.0)
    return float(np.mean(gains))


def eval_stats(ranks, rel_sets):
    """统一口径的五个指标 —— 全部 round 到 4 位，便于对账。"""
    return {"R@5": round(recall_at(ranks, rel_sets, 5), 4),
            "P@5": round(float(np.mean([len(set(r[:5]) & rel) / 5
                                        for r, rel in zip(ranks, rel_sets)])), 4),
            "Hit@5": round(hit_at(ranks, rel_sets, 5), 4),
            "MRR": round(reciprocal_rank(ranks, rel_sets), 4),
            "nDCG@10": round(ndcg_at(ranks, rel_sets, 10), 4)}


def quadrant(doc_idx, q_idx, _=None):
    """把「文档 doc_idx 对查询 q_idx 而言」归到四个象限之一（误差归因用）。"""
    if doc_idx in REL[q_idx]:
        return "①同设备+同缺陷"
    if DOCS.iloc[doc_idx]["设备类型"] == QUERIES.iloc[q_idx]["设备类型"]:
        return "②同设备+异缺陷"
    if DOCS.iloc[doc_idx]["缺陷类型"] == QUERIES.iloc[q_idx]["缺陷类型"]:
        return "③异设备+同缺陷"
    return "④都不沾"


# ---- 全章统一的语料与索引（后面所有任务都在这套口径上比）----
DOCS, QUERIES = stratified_split(load_corpus())
DOC_TOKENS = tokenize_all(DOCS["text"])
QUERY_TOKENS = tokenize_all(QUERIES["text"])
REL = relevant_sets(DOCS, QUERIES)
BM = BM25(DOC_TOKENS)

VEC = TfidfVectorizer(tokenizer=str.split, token_pattern=None, sublinear_tf=True)
XD = VEC.fit_transform([" ".join(d) for d in DOC_TOKENS])      # 文档 TF-IDF 矩阵
XQ = VEC.transform([" ".join(q) for q in QUERY_TOKENS])        # 查询 TF-IDF 矩阵
SIM = (XQ @ XD.T).toarray()                                    # 稀疏点积 = 余弦相似度

## 任务 1：清洗深度 —— 同一批行，两种洗法

注意：**同一批文档行**，只换清洗函数。这是受控实验。

In [ ]:
clean_rows = []
for tag, fn in [("A 只去 HTML+台区", clean_basic), ("B +URL +去空格", clean_full)]:
    dtk = tokenize_all(DOCS["缺陷描述"].map(fn))
    qtk = tokenize_all(QUERIES["缺陷描述"].map(fn))
    engine = BM25(dtk)
    # TODO(1)：算出该清洗口径下的「词表大小」「平均文档长度」，并把 eval_stats 的检索指标一起收进表里
    ____
clean_table = pd.DataFrame(clean_rows).set_index("口径")
print(clean_table.to_string())
# TODO(2)：算出 B 口径相对 A 口径的指标增量（先丢掉「平均文档长度」与「词表」两列，再 round 到 4 位）
gain = ____
clean_gain = ____
assert clean_table.loc["A 只去 HTML+台区", "词表"] == 226
assert clean_table.loc["B +URL +去空格", "词表"] == 70
assert round(clean_table.loc["B +URL +去空格", "平均文档长度"], 3) == 5.575
assert clean_table.loc["A 只去 HTML+台区", "MRR"] == 0.5659
assert clean_table.loc["B +URL +去空格", "MRR"] == 0.6111
print("B − A：", clean_gain)

## 任务 2：无效占位符 —— 平均文档长度被谁拉低了

统计索引里的无效占位符条数，以及剔掉前后的 `avgdl`。

In [ ]:
bad_mask = DOCS["text"].str.strip().isin(BAD_PLACEHOLDERS).to_numpy()
# TODO(3)：统计索引里「无效占位符」的条数，以及剔掉它们前后两版平均文档长度
n_bad = ____
avg_before = ____
kept_tokens = ____
avg_after = ____
assert n_bad == 17
assert round(avg_before, 3) == 5.575
assert round(avg_after, 3) == 5.729
raw_bad = RAW["缺陷描述"].str.strip()
print("原始 CSV 里的无效文本：空白 %d 条 | N/A %d 条 | '-' %d 条 | '无' %d 条"
      % ((raw_bad == "").sum(), (raw_bad == "N/A").sum(), (raw_bad == "-").sum(), (raw_bad == "无").sum()))
print("进入索引的无效占位符：", DOCS.loc[bad_mask, "text"].value_counts().to_dict())
print("零 token 文档数：", sum(1 for d in DOC_TOKENS if len(d) == 0))
print(f"平均文档长度 {avg_before:.3f} → {avg_after:.3f}（+{(avg_after / avg_before - 1) * 100:.1f}%）")

## 任务 3：手写 BM25 第一步 —— 概率型 idf

按 `log(1 + (N − n + 0.5)/(n + 0.5))` 算出 idf 字典，并与脚手架对账。

In [ ]:
dfreq = Counter()
for d in DOC_TOKENS:
    dfreq.update(set(d))
# TODO(4)：按概率型 idf 公式 log(1 + (N − n + 0.5) / (n + 0.5)) 算出 idf 字典，并找出 idf 最高、最低的两个词
idf_ratio = ____
idf_manual = ____
hi_word = ____
lo_word = ____
assert len(idf_manual) == 70
assert all(abs(idf_manual[w] - BM.idf[w]) < 1e-12 for w in idf_manual)
assert dfreq["核实"] == 151 and dfreq["N"] == 5
assert lo_word == "核实" and hi_word in {"A", "N"}
assert round(idf_manual[hi_word], 6) == 4.570014
print(f"词表 {len(idf_manual)} 个词 | N = {len(DOC_TOKENS)} 篇文档")
print(f"idf 最高的词：{sorted([w for w in idf_manual if idf_manual[w] == idf_manual[hi_word]])}"
      f" = {idf_manual[hi_word]:.6f}（df={dfreq[hi_word]}）"
      f" | max() 选中了 {hi_word!r}，这取决于「谁先被遇到」")
print(f"idf 最低：{lo_word} = {idf_manual[lo_word]:.6f}（df={dfreq[lo_word]}）")

## 任务 4：手写 BM25 第二步 —— 给一条查询的一条文档打分

idf × tf 饱和 × 长度归一化，逐词相加。

In [ ]:
i0 = 0
top_doc_0 = BM.rank(QUERY_TOKENS[i0])[0]
dl0 = len(DOC_TOKENS[top_doc_0])
tf0 = BM.tf[top_doc_0]
words0 = sorted(set(QUERY_TOKENS[i0]) & set(tf0))
# TODO(5)：手工复算 top_doc_0 对第 0 条查询的 BM25 分数（idf × tf 饱和 × 长度归一化，逐词相加）
denom0 = ____
contrib = ____
score_manual = ____
assert abs(score_manual - BM.score(QUERY_TOKENS[i0], top_doc_0)) < 1e-9
assert round(score_manual, 4) == 10.7924
assert top_doc_0 == 14 and DOCS.iloc[top_doc_0]["工单编号"] == "WO_0016"
print("查询：", QUERIES.iloc[i0]["text"], "| 真实分组：", QUERIES.iloc[i0]["group"])
print(f"榜首文档 {DOCS.iloc[top_doc_0]['工单编号']}（{DOCS.iloc[top_doc_0]['group']}）"
      f" 得分 {score_manual:.4f} | 文档长度 {dl0} | 全库平均长度 {BM.avgdl:.3f}")
print("逐词贡献：", {w: round(c, 3) for w, c in zip(words0, contrib)})

## 任务 5：分层切分与正样本规模

统计每条查询的正样本数分布。

In [ ]:
# TODO(6)：统计每条查询的正样本数（同分组的文档条数）的 min / 中位 / max / 平均
sizes = ____
size_min, size_med, size_max, size_avg = ____
assert (len(DOCS), len(QUERIES)) == (530, 58)
assert DOCS["group"].nunique() == 40
assert (size_min, size_med, size_max) == (6, 13, 23)
assert size_avg == 14.09
print(f"语料 {len(DOCS)} 篇文档 / {len(QUERIES)} 条查询 / {DOCS['group'].nunique()} 个分组")
print(f"每条查询的正样本数：min {size_min} | 中位 {size_med} | max {size_max} | 平均 {size_avg}")
print("正样本 ≥ 5 的查询占比：%.4f" % float((sizes >= 5).mean()))

## 任务 6：四条检索路线

造出 BM25 / TF-IDF / SVD(64) / RRF 四套完整排序。

In [ ]:
SVD = TruncatedSVD(n_components=64, random_state=42)
# TODO(7)：造出四条路线的完整排序：BM25 / TF-IDF 稀疏向量 / SVD(64) 稠密向量 / RRF 融合
bm_rank = ____
vec_rank = ____
zd = ____
zq = ____
svd_rank = ____
rrf_rank = ____
assert BM.N == 530 and len(bm_rank) == 58 and len(bm_rank[0]) == 530
route_table = pd.DataFrame([{"方法": tag, **eval_stats(rk, REL)}
                            for tag, rk in [("BM25", bm_rank), ("TF-IDF 向量", vec_rank),
                                            ("SVD 向量(64)", svd_rank), ("RRF 融合", rrf_rank)]]).set_index("方法")
print(route_table.to_string())

## 任务 7：指标拆解 —— R@5 为什么这么低

同时算实测 `R@k` 与理论上限，理解“多正样本”的指标含义。

In [ ]:
cover_rows = []
for k in (1, 3, 5, 10, 20):
    # TODO(8)：算出实测 R@k / Hit@k，以及 R@k 的「理论上限」（正样本被截断到 k 条时的最大 Recall）
    ____
cover_table = pd.DataFrame(cover_rows).set_index("k")
cover_table["占上限%"] = (cover_table["R@k"] / cover_table["上限"] * 100).round(1)
print(cover_table.to_string())

## 任务 8：SVD 降到多少维才不亏

扫 16 / 32 / 64 维的解释方差比。

In [ ]:
var_rows = []
for nc in (16, 32, 64):
    # TODO(9)：用 nc 个主成分拟合 SVD，取出解释方差比之和（保留 6 位小数）
    svd = ____
    ____
var_table = pd.DataFrame(var_rows).set_index("维度")
print(var_table.to_string())
assert var_table.loc[64, "解释方差比"] == 1.0
assert var_table.loc[16, "解释方差比"] == 0.588147

## 任务 9：BM25 的 k1 / b 扫描

12 组网格，看 `Hit@5` 和 `MRR` 的最优点是否一致。

In [ ]:
sweep_rows = []
for k1 in (0.9, 1.5, 2.0):
    for b in (0.0, 0.5, 0.75, 1.0):
        # TODO(10)：用该 (k1, b) 重建 BM25、给全部查询排序，把 Hit@5 与 MRR 收进表里
        eng = ____
        st = ____
        ____
sweep_table = pd.DataFrame(sweep_rows).set_index(["k1", "b"])
print(sweep_table.to_string())
assert sweep_table["Hit@5"].max() == 0.8103
assert sweep_table["MRR"].max() == 0.6611

## 任务 10：融合与重排的收益

RRF 的 `k` + 重排的向量权重，各扫一遍。**注意两路分数要先归一化。**

In [ ]:
k_rows = []
for k_rrf in (10, 30, 60, 100):
    # TODO(11)：用该 k 融合 BM25 与 TF-IDF 两路排序，算出 Hit@5 与 MRR
    st = ____
    ____
print(pd.DataFrame(k_rows).set_index("RRF k").to_string())

rerank_rows = []
for w_vec in (0.0, 0.3, 0.5, 0.7, 1.0):
    reranked = []
    for i in range(len(QUERIES)):
        cand = bm_rank[i][:50]
        b_max = max(BM.score(QUERY_TOKENS[i], d) for d in cand) or 1.0
        v_max = max(SIM[i][d] for d in cand) or 1.0
        # TODO(12)：把两路分数各自归一化到 [0,1] 后按向量权重 w_vec 加权，重排 top-50 候选，再接上第 50 名之后的原序
        key = ____
        ____
    rerank_rows.append({"向量权重": w_vec, **eval_stats(reranked, REL)})
rerank_table = pd.DataFrame(rerank_rows).set_index("向量权重")
print(rerank_table.to_string())
assert rerank_table["MRR"].max() == 0.6321
assert rerank_table.loc[0.0, "MRR"] == 0.6111

## 任务 11：误差归因 —— top-5 里的错，错在哪一类

用四象限拆开 top-5，找出最大的一类错误。

In [ ]:
attr_rows = []
for tag, ranks in [("BM25", bm_rank), ("TF-IDF 向量", vec_rank)]:
    labels = []
    for i, r in enumerate(ranks):
        # TODO(13)：用 quadrant() 把该查询 top-5 里的每个文档翻译成象限标签，追加进 labels
        ____
    cnt = Counter(labels)
    tot = sum(cnt.values())
    attr_rows.append({"方法": tag, **{k: round(v / tot * 100, 1) for k, v in sorted(cnt.items())}})
attr_table = pd.DataFrame(attr_rows).set_index("方法")
print(attr_table.to_string())
assert attr_table.loc["BM25", "③异设备+同缺陷"] == 40.0
assert attr_table.loc["BM25", "④都不沾"] == 1.7

## 任务 12：定向改进 —— 拿误差归因换指标

按任务 11 的归因做两件事：

1. **设备元数据硬过滤**：只让同设备类型的文档进榜（应对「异设备+同缺陷」）；
2. **在硬过滤的基础上给缺陷词 ×2 加权**（放大信号词、压制模板词）。

> ⚠️ 两阶段叠加才是最优解，单做一步收益差很多。

In [ ]:
improved_rows = [{"策略": "基线 BM25", **eval_stats(bm_rank, REL)}]

dev_filter_rank = []
for i in range(len(QUERIES)):
    dev = QUERIES.iloc[i]["设备类型"]
    # TODO(14)：把同设备类型的文档按原 BM25 名次提到前面，异设备文档整体接在后段
    same_dev = ____
    other_dev = ____
    ____
improved_rows.append({"策略": "设备元数据硬过滤", **eval_stats(dev_filter_rank, REL)})

two_stage_rank = []
for i in range(len(QUERIES)):
    dev = QUERIES.iloc[i]["设备类型"]
    cand = [j for j in range(len(DOCS)) if DOCS.iloc[j]["设备类型"] == dev]
    # TODO(15)：在同设备候选上按「缺陷词 ×2」重新打分排序，其余文档接在后段
    scores = ____
    ordered = ____
    rest = ____
    ____
improved_rows.append({"策略": "硬过滤 + 缺陷词 ×2", **eval_stats(two_stage_rank, REL)})
improved_table = pd.DataFrame(improved_rows).set_index("策略")
print(improved_table.to_string())
assert improved_table.loc["硬过滤 + 缺陷词 ×2", "MRR"] == 0.9345
assert improved_table.loc["硬过滤 + 缺陷词 ×2", "Hit@5"] == 0.9655

## 任务 13：拒答 —— 检索不到就别硬答

先看三类查询的榜首分数分布，再找阈值缝隙。

In [ ]:
GENERIC = ["今天长沙天气怎么样", "帮我订一张去北京的机票", "这道数学题怎么解",
           "食堂中午几点开饭", "如何申请年假", "推荐一部好看的电影"]
TOPICAL = ["变压器的工作原理是什么", "电力设备的保养周期是多久", "绝缘子的国家标准是什么",
           "如何填写工单编号", "断路器的作用有哪些", "母线电压等级怎么选"]

hit_top = [BM.score(QUERY_TOKENS[i], bm_rank[i][0]) for i in range(len(QUERIES))]
gen_top = []
for text in GENERIC:
    # TODO(16)：给这条「完全无关」的查询排序，取榜首分数追加进 gen_top
    q_tokens = ____
    ____
topical_top = []
for text in TOPICAL:
    # TODO(17)：给这条「含语料词但意图不同」的查询排序，取榜首分数追加进 topical_top
    q_tokens = ____
    ____

score_rows = []
for tag, group in [("真命中查询", hit_top), ("完全无关", gen_top), ("含语料词但意图不同", topical_top)]:
    score_rows.append({"查询类型": tag, "min": round(min(group), 3),
                       "中位": round(float(np.median(group)), 3), "max": round(max(group), 3)})
print(pd.DataFrame(score_rows).set_index("查询类型").to_string())

thr_rows = []
for thr in (1.0, 2.0, 3.0, 4.0):
    # TODO(18)：统计该阈值下「真命中被保留 / 无关查询被拒 / 沾边查询被拒」的条数
    ____
print(pd.DataFrame(thr_rows).set_index("阈值").to_string())
assert max(gen_top) == 0.0
assert round(float(np.median(hit_top)), 3) == 10.805
assert sum(1 for s in topical_top if s < 4.0) == 6
assert sum(1 for s in topical_top if s < 3.0) == 4

## 任务 14：prompt 拼装与引用

拼出带编号引用、含结构化字段、以问题收尾、并写明“资料不足说无法确定”的 prompt。

In [ ]:
i_p = 0
picked = bm_rank[i_p][:3]
# TODO(19)：拼出带编号引用的 prompt：参考资料每条形如「[1] 工单 WO_0016｜设备：绝缘子｜缺陷：渗漏油｜等级：危急」，
#      再附上该条的描述、用户问题、以及「只依据资料作答 + 标注引用编号 + 资料不足说无法确定」的作答要求
ref_lines = ____
desc_lines = ____
ref_blocks = ____
prompt = ____
assert prompt.startswith("【参考资料】")
assert "[1]" in prompt and "[3]" in prompt
assert "WO_0016" in prompt
assert "无法确定" in prompt
print(prompt)
print("\n--- 检索到的 top-3 与真实分组的关系 ---")
for pos, d in enumerate(picked, 1):
    print(f"  [{pos}] {DOCS.iloc[d]['group']} | 与查询同分组：{d in REL[i_p]}")

## 自查清单

- [ ] 能说出"换语料必须固定评测集"，并解释任务 1 为什么是受控实验
- [ ] 能背出 BM25 的三块（idf / tf 饱和 / 长度归一化）
- [ ] 知道 idf 的 `+1` 是干什么的（保证非负）
- [ ] 知道 `avgdl` 污染影响的是**全库**分数
- [ ] 会算 `R@k` 的上限，并知道多正样本下 `R@5` 低不代表检索差
- [ ] 能解释 SVD(64) 为什么和 TF-IDF 指标相同
- [ ] 能画出误差归因四象限，说出最大错误是哪一类
- [ ] 知道任务 12 的两招为什么有效（改候选集 + 放大信号词）
- [ ] 知道 BM25 零分是天然的拒答信号
- [ ] 能复述改进顺序：**修数据 → 定口径 → 看归因 → 定向改进 → 调算法**